Original multi-agent business-insights experiment
Historical experiment retained for provenance. Use the repository README for the maintained entry point.


In [ ]:
# -*- coding: utf-8 -*-
"""agentpractice4.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1hbelBV5hKfvDhoWknacDm6qZY3aEl-JC

graph TD
  Start([User Input Form])
  Start --> F1[Feedback Summarizer Agent]
  Start --> S1[Sales Analyzer Agent]
  F1 --> I1[Insights Generator Agent]
  S1 --> I1
  I1 --> E1[Email Generator Agent]
  I1 --> M1[Marketing Idea Generator Agent]
  E1 --> Output1([Generated Email])
  M1 --> Output2([Marketing Ideas])
  F1 --> Output3([Customer Feedback Summary])
  S1 --> Output4([Sales Insight Summary])
"""

!pip install langchain openai langgraph pandas langchain.community
!pip install langchain-openai

import os



import os
from typing import TypedDict, Annotated , List
import operator

from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END

# Define the State for our LangGraph
class AgentState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        customer_feedback: Raw customer feedback (text or CSV path).
        sales_summaries: Raw sales summaries (CSV/Excel path or text).
        business_type: Type of business (e.g., "ecommerce", "SaaS").
        email_tone: Preferred email tone (e.g., "formal", "friendly", "persuasive").
        feedback_summary: Summarized customer pain points and praise.
        sales_insights: Sales trends, top products, churn signals, seasonal changes.
        key_issues: Key issues identified from feedback and sales.
        opportunities: Opportunities identified from feedback and sales.
        customer_behavior_profile: Profile of customer behavior.
        generated_email: The generated email content.
        marketing_ideas: Suggested marketing campaign ideas, content hooks, product improvements.
        messages: Conversation history for the LLM.
    """

    customer_feedback: str
    sales_summaries: str
    business_type: str
    email_tone: str
    feedback_summary: Annotated[str, operator.add]
    sales_insights: Annotated[str, operator.add]
    key_issues: Annotated[str, operator.add]
    opportunities: Annotated[str, operator.add]
    customer_behavior_profile: Annotated[str, operator.add]
    generated_email: Annotated[str, operator.add]
    marketing_ideas: Annotated[str, operator.add]
    messages: Annotated[List[BaseMessage], operator.add]

# Initialize the Language Model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# 1. Feedback Summarizer Agent
def feedback_summarizer_agent(state: AgentState):
    """
    Summarizes customer feedback, identifying pain points and praise.
    Handles both text and CSV input.
    """
    print("---FEEDBACK SUMMARIZER AGENT---")
    customer_feedback = state.get("customer_feedback", "")
    messages = state.get("messages", [])

    if not customer_feedback:
        print("No customer feedback provided.")
        return {"feedback_summary": "No feedback provided.", "messages": messages + [AIMessage(content="No customer feedback provided for summarization.")]}

    feedback_text = ""
    if customer_feedback.lower().endswith(".csv"):
        try:
            loader = CSVLoader(file_path=customer_feedback)
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            splits = text_splitter.split_documents(docs)
            feedback_text = "\n".join([s.page_content for s in splits])
        except Exception as e:
            print(f"Error loading CSV feedback: {e}")
            feedback_text = customer_feedback # Treat as raw text if CSV loading fails
    else:
        feedback_text = customer_feedback

    summarize_prompt_template = """
    You are an expert at summarizing customer feedback.
    Analyze the following customer feedback and extract the main pain points and areas of praise.
    Organize your summary clearly with "Pain Points:" and "Praise:".

    Customer Feedback:
    "{text}"

    Summary:
    """
    prompt = PromptTemplate(template=summarize_prompt_template, input_variables=["text"])
    summarize_chain = load_summarize_chain(llm, chain_type="stuff", prompt=prompt)

    try:
        # LangChain's load_summarize_chain expects a list of Document objects.
        # If feedback_text is a string, we need to convert it.
        from langchain_core.documents import Document
        docs_for_chain = [Document(page_content=feedback_text)]
        summary_result = summarize_chain.run(docs_for_chain)
        print(f"Feedback Summary: {summary_result}")
        return {"feedback_summary": summary_result, "messages": messages + [AIMessage(content=f"Feedback summarized: {summary_result}")]}
    except Exception as e:
        print(f"Error summarizing feedback: {e}")
        return {"feedback_summary": f"Error summarizing feedback: {e}", "messages": messages + [AIMessage(content=f"Error summarizing feedback: {e}")]}

# 2. Sales Analyzer Agent
def sales_analyzer_agent(state: AgentState):
    """
    Parses sales trends from sales summaries, highlighting top-performing products,
    customer churn signals, and seasonal changes. Handles both text and CSV/Excel input.
    """
    print("---SALES ANALYZER AGENT---")
    sales_summaries = state.get("sales_summaries", "")
    messages = state.get("messages", [])

    if not sales_summaries:
        print("No sales summaries provided.")
        return {"sales_insights": "No sales data provided.", "messages": messages + [AIMessage(content="No sales summaries provided for analysis.")]}

    sales_text = ""
    if sales_summaries.lower().endswith(".csv"):
        try:
            loader = CSVLoader(file_path=sales_summaries)
            docs = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            splits = text_splitter.split_documents(docs)
            sales_text = "\n".join([s.page_content for s in splits])
        except Exception as e:
            print(f"Error loading CSV sales data: {e}")
            sales_text = sales_summaries # Treat as raw text if CSV loading fails
    else:
        sales_text = sales_summaries

    sales_analysis_prompt_template = """
    You are an expert sales analyst.
    Analyze the following sales data and identify:
    1. Top-performing products/services.
    2. Potential customer churn signals (e.g., declining purchases, reduced engagement).
    3. Any noticeable seasonal changes or trends.
    Provide your insights clearly.

    Sales Data:
    "{text}"

    Sales Insights:
    """
    prompt = PromptTemplate(template=sales_analysis_prompt_template, input_variables=["text"])

    try:
        response = llm.invoke(prompt.format(text=sales_text))
        insights = response.content
        print(f"Sales Insights: {insights}")
        return {"sales_insights": insights, "messages": messages + [AIMessage(content=f"Sales analyzed: {insights}")]}
    except Exception as e:
        print(f"Error analyzing sales: {e}")
        return {"sales_insights": f"Error analyzing sales: {e}", "messages": messages + [AIMessage(content=f"Error analyzing sales: {e}")]}

# 3. Insights Generator Agent
def insights_generator_agent(state: AgentState):
    """
    Merges feedback and sales insights to generate key issues, opportunities,
    and a customer behavior profile.
    """
    print("---INSIGHTS GENERATOR AGENT---")
    feedback_summary = state.get("feedback_summary", "")
    sales_insights = state.get("sales_insights", "")
    business_type = state.get("business_type", "general business")
    messages = state.get("messages", [])

    insights_prompt_template = """
    You are a strategic business analyst.
    Combine the following customer feedback summary and sales insights to generate:
    1. Key Issues: Major problems or areas for improvement.
    2. Opportunities: Potential areas for growth or new initiatives.
    3. Customer Behavior Profile: A description of typical customer actions and preferences.

    Consider the business type: {business_type}.

    Customer Feedback Summary:
    "{feedback_summary}"

    Sales Insights:
    "{sales_insights}"

    Combined Insights:
    """
    prompt = PromptTemplate(
        template=insights_prompt_template,
        input_variables=["feedback_summary", "sales_insights", "business_type"]
    )

    try:
        response = llm.invoke(prompt.format(
            feedback_summary=feedback_summary,
            sales_insights=sales_insights,
            business_type=business_type
        ))
        combined_insights = response.content

        # Attempt to parse the combined insights into structured fields
        key_issues = "Not specified."
        opportunities = "Not specified."
        customer_behavior_profile = "Not specified."

        if "Key Issues:" in combined_insights:
            parts = combined_insights.split("Key Issues:", 1)
            key_issues_part = parts[1]
            if "Opportunities:" in key_issues_part:
                key_issues = key_issues_part.split("Opportunities:", 1)[0].strip()
            else:
                key_issues = key_issues_part.strip()

        if "Opportunities:" in combined_insights:
            parts = combined_insights.split("Opportunities:", 1)
            opportunities_part = parts[1]
            if "Customer Behavior Profile:" in opportunities_part:
                opportunities = opportunities_part.split("Customer Behavior Profile:", 1)[0].strip()
            else:
                opportunities = opportunities_part.strip()

        if "Customer Behavior Profile:" in combined_insights:
            customer_behavior_profile = combined_insights.split("Customer Behavior Profile:", 1)[1].strip()

        print(f"Key Issues: {key_issues}")
        print(f"Opportunities: {opportunities}")
        print(f"Customer Behavior Profile: {customer_behavior_profile}")

        return {
            "key_issues": key_issues,
            "opportunities": opportunities,
            "customer_behavior_profile": customer_behavior_profile,
            "messages": messages + [AIMessage(content=f"Insights generated. Key Issues: {key_issues}, Opportunities: {opportunities}, Customer Behavior Profile: {customer_behavior_profile}")]
        }
    except Exception as e:
        print(f"Error generating insights: {e}")
        return {
            "key_issues": f"Error: {e}",
            "opportunities": f"Error: {e}",
            "customer_behavior_profile": f"Error: {e}",
            "messages": messages + [AIMessage(content=f"Error generating insights: {e}")]
        }

# 4. Email Generator Agent
def email_generator_agent(state: AgentState):
    """
    Generates an email based on insights, preferred tone, and business type.
    """
    print("---EMAIL GENERATOR AGENT---")
    key_issues = state.get("key_issues", "")
    opportunities = state.get("opportunities", "")
    customer_behavior_profile = state.get("customer_behavior_profile", "")
    email_tone = state.get("email_tone", "friendly")
    business_type = state.get("business_type", "general business")
    messages = state.get("messages", [])

    email_prompt_template = """
    You are an expert email copywriter.
    Generate an email based on the following insights.
    The email should be suitable for a {business_type} business.
    The tone should be {email_tone}.

    Insights:
    Key Issues: {key_issues}
    Opportunities: {opportunities}
    Customer Behavior Profile: {customer_behavior_profile}

    Email Type (choose one or suggest a new one based on context, e.g., "Customer Apology", "Follow-up", "Marketing Newsletter"):
    {email_type_suggestion}

    Generated Email:
    """

    # Determine a suitable email type based on key issues
    email_type_suggestion = "Marketing Newsletter"
    if "apology" in key_issues.lower() or "issue" in key_issues.lower():
        email_type_suggestion = "Customer Apology"
    elif "follow-up" in opportunities.lower():
        email_type_suggestion = "Follow-up"

    prompt = PromptTemplate(
        template=email_prompt_template,
        input_variables=["key_issues", "opportunities", "customer_behavior_profile", "email_tone", "business_type", "email_type_suggestion"]
    )

    try:
        response = llm.invoke(prompt.format(
            key_issues=key_issues,
            opportunities=opportunities,
            customer_behavior_profile=customer_behavior_profile,
            email_tone=email_tone,
            business_type=business_type,
            email_type_suggestion=email_type_suggestion
        ))
        generated_email = response.content
        print(f"Generated Email: {generated_email}")
        return {"generated_email": generated_email, "messages": messages + [AIMessage(content=f"Email generated: {generated_email}")]}
    except Exception as e:
        print(f"Error generating email: {e}")
        return {"generated_email": f"Error generating email: {e}", "messages": messages + [AIMessage(content=f"Error generating email: {e}")]}

# 5. Marketing Idea Generator Agent
def marketing_idea_generator_agent(state: AgentState):
    """
    Suggests marketing campaign ideas, content hooks, and product improvements
    based on insights and preferred tone.
    """
    print("---MARKETING IDEA GENERATOR AGENT---")
    key_issues = state.get("key_issues", "")
    opportunities = state.get("opportunities", "")
    customer_behavior_profile = state.get("customer_behavior_profile", "")
    email_tone = state.get("email_tone", "friendly") # Using email_tone as a general tone
    business_type = state.get("business_type", "general business")
    messages = state.get("messages", [])

    marketing_prompt_template = """
    You are a creative marketing strategist.
    Based on the following insights for a {business_type} business, suggest:
    1. Campaign Ideas: Innovative marketing campaign concepts.
    2. Content Hooks: Engaging ideas for marketing content (e.g., blog posts, social media).
    3. Product Improvements: Suggestions for enhancing products/services.

    Consider the overall tone should be {email_tone}.

    Insights:
    Key Issues: {key_issues}
    Opportunities: {opportunities}
    Customer Behavior Profile: {customer_behavior_profile}

    Marketing Ideas:
    """
    prompt = PromptTemplate(
        template=marketing_prompt_template,
        input_variables=["key_issues", "opportunities", "customer_behavior_profile", "email_tone", "business_type"]
    )

    try:
        response = llm.invoke(prompt.format(
            key_issues=key_issues,
            opportunities=opportunities,
            customer_behavior_profile=customer_behavior_profile,
            email_tone=email_tone,
            business_type=business_type
        ))
        marketing_ideas = response.content
        print(f"Marketing Ideas: {marketing_ideas}")
        return {"marketing_ideas": marketing_ideas, "messages": messages + [AIMessage(content=f"Marketing ideas generated: {marketing_ideas}")]}
    except Exception as e:
        print(f"Error generating marketing ideas: {e}")
        return {"marketing_ideas": f"Error generating marketing ideas: {e}", "messages": messages + [AIMessage(content=f"Error generating marketing ideas: {e}")]}

# --- Build the LangGraph ---

workflow = StateGraph(AgentState)

# Add nodes for each agent
workflow.add_node("feedback_summarizer", feedback_summarizer_agent)
workflow.add_node("sales_analyzer", sales_analyzer_agent)
workflow.add_node("insights_generator", insights_generator_agent)
workflow.add_node("email_generator", email_generator_agent)
workflow.add_node("marketing_idea_generator", marketing_idea_generator_agent)

# Set the entry point
workflow.set_entry_point("feedback_summarizer")

# Define edges
workflow.add_edge("feedback_summarizer", "sales_analyzer") # F1 -> S1 (sequential for simplicity, can be parallel)
workflow.add_edge("sales_analyzer", "insights_generator") # S1 -> I1
workflow.add_edge("insights_generator", "email_generator") # I1 -> E1
workflow.add_edge("insights_generator", "marketing_idea_generator") # I1 -> M1

# Define the end points
workflow.add_edge("email_generator", END)
workflow.add_edge("marketing_idea_generator", END)

# Compile the graph
app = workflow.compile()

# --- Test Input and Execution ---

# Example User Input (you can modify these)
# For CSV files, ensure they are uploaded to your Colab environment
# or provide a direct URL if accessible.

# Example 1: Text-based input
user_input_text = {
    "customer_feedback": """
    The new software update is very buggy. I keep getting error messages when trying to save documents.
    Customer support was quick to respond, which was good.
    The old interface was more intuitive, this new one is confusing.
    I love the new dark mode feature, it's great for my eyes.
    """,
    "sales_summaries": """
    Q1 Sales Report:
    Product A: $150,000 (down 10% from last quarter)
    Product B: $200,000 (up 5% from last quarter)
    Product C: $80,000 (new product, performing well)
    Customer retention rate for Product A users has dropped by 3% this quarter.
    Sales typically peak in December due to holiday season.
    """,
    "business_type": "SaaS",
    "email_tone": "formal"
}

# Example 2: Placeholder for CSV input (you would need to create these files)
# To test with CSVs:
# 1. Create 'customer_feedback.csv' and 'sales_summaries.csv' in your Colab environment.
# 2. Populate them with relevant data.
# 3. Uncomment and use this input dictionary.

# user_input_csv = {
#     "customer_feedback": "customer_feedback.csv",
#     "sales_summaries": "sales_summaries.csv",
#     "business_type": "ecommerce",
#     "email_tone": "friendly"
# }

# Function to run the graph and print results
def run_agent_graph(input_data):
    print("\n--- RUNNING AGENT GRAPH ---")
    inputs = {
        "customer_feedback": input_data["customer_feedback"],
        "sales_summaries": input_data["sales_summaries"],
        "business_type": input_data["business_type"],
        "email_tone": input_data["email_tone"],
        "feedback_summary": "", # Initialize
        "sales_insights": "",    # Initialize
        "key_issues": "",        # Initialize
        "opportunities": "",     # Initialize
        "customer_behavior_profile": "", # Initialize
        "generated_email": "",   # Initialize
        "marketing_ideas": "",   # Initialize
        "messages": [HumanMessage(content="Starting analysis based on provided data.")]
    }
    for s in app.stream(inputs):
        print(s)
        print("---")

    final_state = app.invoke(inputs)
    print("\n--- FINAL STATE ---")
    print(f"Customer Feedback Summary:\n{final_state['feedback_summary']}")
    print(f"\nSales Insight Summary:\n{final_state['sales_insights']}")
    print(f"\nKey Issues:\n{final_state['key_issues']}")
    print(f"\nOpportunities:\n{final_state['opportunities']}")
    print(f"\nCustomer Behavior Profile:\n{final_state['customer_behavior_profile']}")
    print(f"\nGenerated Email:\n{final_state['generated_email']}")
    print(f"\nMarketing Ideas:\n{final_state['marketing_ideas']}")
    return final_state

# Run the graph with the text-based input
final_output = run_agent_graph(user_input_text)

import gradio as gr

# Define the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # Business Insights and Email Generator
        Analyze customer feedback and sales data to generate key business insights,
        a customer behavior profile, a tailored email, and marketing ideas.
        """
    )
    with gr.Row():
        with gr.Column():
            gr.Markdown("## Customer Feedback")
            customer_feedback_text = gr.Textbox(
                label="Enter Customer Feedback (or upload CSV below)",
                lines=5,
                placeholder="e.g., 'The new software update is buggy. Customer support was quick. Old interface was better.'"
            )
            customer_feedback_file = gr.File(
                label="Upload Customer Feedback CSV",
                type="filepath", # Use 'filepath' for temporary file paths
                file_types=[".csv"]
            )
            gr.Markdown("## Sales Summaries")
            sales_summaries_text = gr.Textbox(
                label="Enter Sales Summaries (or upload CSV below)",
                lines=5,
                placeholder="e.g., 'Q1 Sales: Product A down 10%, Product B up 5%. Product C new, performing well.'"
            )
            sales_summaries_file = gr.File(
                label="Upload Sales Summaries CSV",
                type="filepath", # Use 'filepath' for temporary file paths
                file_types=[".csv"]
            )
            gr.Markdown("## Business Details")
            business_type = gr.Dropdown(
                label="Business Type",
                choices=["ecommerce", "SaaS", "retail", "service-based", "consulting", "other"],
                value="SaaS"
            )
            email_tone = gr.Radio(
                label="Preferred Email Tone",
                choices=["formal", "friendly", "persuasive", "empathetic"],
                value="formal"
            )
            submit_btn = gr.Button("Generate Insights & Content", variant="primary")

        with gr.Column():
            gr.Markdown("## Generated Outputs")
            feedback_summary_output = gr.Textbox(label="Customer Feedback Summary", lines=3, interactive=False)
            sales_insights_output = gr.Textbox(label="Sales Insights", lines=3, interactive=False)
            key_issues_output = gr.Textbox(label="Key Issues", lines=3, interactive=False)
            opportunities_output = gr.Textbox(label="Opportunities", lines=3, interactive=False)
            customer_behavior_profile_output = gr.Textbox(label="Customer Behavior Profile", lines=3, interactive=False)
            generated_email_output = gr.Textbox(label="Generated Email", lines=10, interactive=False)
            marketing_ideas_output = gr.Textbox(label="Marketing Ideas", lines=7, interactive=False)

    submit_btn.click(
        process_business_data,
        inputs=[
            customer_feedback_text, customer_feedback_file,
            sales_summaries_text, sales_summaries_file,
            business_type, email_tone
        ],
        outputs=[
            feedback_summary_output, sales_insights_output,
            key_issues_output, opportunities_output,
            customer_behavior_profile_output, generated_email_output,
            marketing_ideas_output
        ]
    )

# Launch the Gradio app
demo.launch(debug=True, share=True) # share=True generates a public link for easy sharing in Colab

def process_business_data(customer_feedback_text, customer_feedback_file, sales_summaries_text, sales_summaries_file, business_type, email_tone):
    """
    Processes the user input from the Gradio interface, runs the LangGraph,
    and returns the results for display in the Gradio output fields.
    """
    # Determine the actual feedback input: file takes precedence over text
    customer_feedback_input = customer_feedback_file if customer_feedback_file else customer_feedback_text

    # Determine the actual sales input: file takes precedence over text
    sales_summaries_input = sales_summaries_file if sales_summaries_file else sales_summaries_text

    inputs = {
        "customer_feedback": customer_feedback_input,
        "sales_summaries": sales_summaries_input,
        "business_type": business_type,
        "email_tone": email_tone,
        "feedback_summary": "", # Initialize
        "sales_insights": "",    # Initialize
        "key_issues": "",        # Initialize
        "opportunities": "",     # Initialize
        "customer_behavior_profile": "", # Initialize
        "generated_email": "",   # Initialize
        "marketing_ideas": "",   # Initialize
        "messages": [HumanMessage(content="Starting analysis based on provided data.")]
    }

    # Run the LangGraph
    # We will use app.invoke for simplicity in the Gradio interface,
    # as Gradio handles the streaming output separately if needed.
    final_state = app.invoke(inputs)

    # Extract the outputs
    feedback_summary = final_state.get("feedback_summary", "N/A")
    sales_insights = final_state.get("sales_insights", "N/A")
    key_issues = final_state.get("key_issues", "N/A")
    opportunities = final_state.get("opportunities", "N/A")
    customer_behavior_profile = final_state.get("customer_behavior_profile", "N/A")
    generated_email = final_state.get("generated_email", "N/A")
    marketing_ideas = final_state.get("marketing_ideas", "N/A")

    return (
        feedback_summary,
        sales_insights,
        key_issues,
        opportunities,
        customer_behavior_profile,
        generated_email,
        marketing_ideas
    )